In [24]:
from ultralytics import YOLO, RTDETR

import cv2
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

In [ ]:
yolov8 = YOLO(
    r"D:\Training + AI\PPE Train\Benchmark\YOLOv8\yolov8_best.pt"
)

yolo26 = YOLO(
    r"D:\Training + AI\PPE Train\Benchmark\YOLO26\yolo26_best.pt"
)

rtdetr = RTDETR(
    r"D:\Training + AI\PPE Train\Benchmark\RT-DETR\rtdetr_best.pt"
)

print("Loaded all models.")

In [ ]:
TEST_DIR = Path(
    r"D:\Training + AI\PPE Train\datasets\PPE-3\test\images"
)

image_files = sorted(
    list(TEST_DIR.glob("*.jpg"))
)

print(len(image_files))

In [ ]:
candidate_images = []

for img_path in tqdm(image_files):

    r1 = yolov8.predict(
        str(img_path),
        conf=0.25,
        verbose=False
    )[0]

    r2 = yolo26.predict(
        str(img_path),
        conf=0.25,
        verbose=False
    )[0]

    r3 = rtdetr.predict(
        str(img_path),
        conf=0.25,
        verbose=False
    )[0]

    cls1 = sorted(r1.boxes.cls.cpu().numpy().astype(int).tolist())
    cls2 = sorted(r2.boxes.cls.cpu().numpy().astype(int).tolist())
    cls3 = sorted(r3.boxes.cls.cpu().numpy().astype(int).tolist())

    score = len(
        set(cls1) ^ set(cls2)
    ) + len(
        set(cls1) ^ set(cls3)
    )

    candidate_images.append(
        (
            score,
            str(img_path)
        )
    )

In [ ]:
candidate_images.sort(
    reverse=True
)

candidate_images[:20]

In [29]:
def compare_models(img_path, save_path=None):

    img = cv2.imread(str(img_path))

    if img is None:
        print(f"Cannot read image: {img_path}")
        return

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    r1 = yolov8.predict(img_path, conf=0.25, verbose=False)[0]
    r2 = yolo26.predict(img_path, conf=0.25, verbose=False)[0]
    r3 = rtdetr.predict(img_path, conf=0.25, verbose=False)[0]

    v8_img = cv2.cvtColor(r1.plot(), cv2.COLOR_BGR2RGB)
    y26_img = cv2.cvtColor(r2.plot(), cv2.COLOR_BGR2RGB)
    rt_img = cv2.cvtColor(r3.plot(), cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 4, figsize=(22, 8))

    ax[0].imshow(img_rgb)
    ax[0].set_title("Original")

    ax[1].imshow(v8_img)
    ax[1].set_title(f"YOLOv8 ({len(r1.boxes)})")

    ax[2].imshow(y26_img)
    ax[2].set_title(f"YOLO26 ({len(r2.boxes)})")

    ax[3].imshow(rt_img)
    ax[3].set_title(f"RT-DETR ({len(r3.boxes)})")

    for a in ax:
        a.axis("off")

    plt.tight_layout()

    if save_path:
        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()
    plt.close(fig)

In [ ]:
for score, img_path in candidate_images[:20]:

    print("="*80)
    print(score)
    print(img_path)

    compare_models(img_path)

In [ ]:
score, img_path = candidate_images[0]

print(score)
print(img_path)

compare_models(img_path)

In [ ]:
TOP_K = 20

for i in range(TOP_K):

    score, img_path = candidate_images[i]

    print("=" * 80)
    print(f"Rank: {i+1}")
    print(f"Score: {score}")
    print(img_path)

    compare_models(img_path)

In [ ]:
from pathlib import Path

SAVE_DIR = Path(
    r"D:\Training + AI\PPE Train\Error Analysis Images"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TOP_K = 20

for i in range(TOP_K):

    score, img_path = candidate_images[i]

    save_file = SAVE_DIR / f"Failure_{i+1:02d}_Score_{score}.png"

    print("=" * 80)
    print(f"Rank: {i+1}")
    print(f"Score: {score}")
    print(img_path)
    print(f"Saving -> {save_file}")

    compare_models(
        img_path,
        save_path=str(save_file)
    )